In [ ]:
pip install pandas

In [2]:
import matplotlib.pyplot as plt
from statistics import correlation

from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, brier_score_loss

In [3]:
import random

import numpy as np
import pandas as pd

def model_function(
    train_path="training.csv",
    test_path="testing.csv",
    train_out="train_modified.csv",
    test_out="test_modified.csv",
):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    league = train["outcome"].mean()
    strength = 80

    g = train.groupby("shooter_id")["outcome"]
    made = train["shooter_id"].map(g.sum())
    n = train["shooter_id"].map(g.count())
    train["shooter_rate"] = (made - train["outcome"] + strength * league) / (n - 1 + strength)

    shooter_rate_map = (g.sum() + strength * league) / (g.count() + strength)
    test["shooter_rate"] = test["shooter_id"].map(shooter_rate_map).fillna(league)

    month_map = {
        "JAN": "1", "FEB": "2", "MAR": "3", "APR": "4",
        "MAY": "5", "JUN": "6", "JUL": "7", "AUG": "8",
        "SEP": "9", "OCT": "10", "NOV": "11", "DEC": "12",
    }
    season_map = {"fe055": "1", "2676a": "2"}
    contested_map = {False: "0", True: "1", "False": "0", "True": "1"}
    shot_map = {
        "jumper": "1", "layup": "2", "dunk": "3", "post": "4",
        "floater": "5", "tip": "6", "heave": "7",
    }
    gamestate_map = {
        "TRANSITION": "1", "HALFCOURT": "2", "DIRECTSECONDCHANCE": "3",
    }
    three_map = {False: "0", True: "1", "False": "0", "True": "1"}

    def encode_ids(source_column):
        values = pd.unique(train[source_column].dropna())
        numbers = random.sample(range(1, len(values) + 2), len(values))
        return dict(zip(values, numbers))

    id_maps = {
        "shooter_id": encode_ids("shooter_id"),
        "closestdef_id": encode_ids("closestdef_id"),
        "off_team_id": encode_ids("off_team_id"),
        "def_team_id": encode_ids("def_team_id"),
        "contester1_id": encode_ids("contester1_id"),
    }

    def apply_changes(df):
        df = df.copy()

        df["Month_Encoded"] = df["month"].map(month_map)
        df = df.drop(columns=["month"])

        df["shooter"] = df["shooter_id"].map(id_maps["shooter_id"])
        df = df.drop(columns=["shooter_id"])

        df["defender"] = df["closestdef_id"].map(id_maps["closestdef_id"])
        df = df.drop(columns=["closestdef_id"])

        df["offense"] = df["off_team_id"].map(id_maps["off_team_id"])
        df = df.drop(columns=["off_team_id"])

        df["defense"] = df["def_team_id"].map(id_maps["def_team_id"])
        df = df.drop(columns=["def_team_id"])

        df["Season_Encoded"] = df["season_id"].map(season_map)
        df = df.drop(columns=["season_id"])

        df = df.drop(columns=["contester3_id", "contester4_id", "distcont3", "distcont4"])

        df["Contested_Encoded"] = df["contested"].map(contested_map)
        df = df.drop(columns=["contested"])

        df["contester"] = df["contester1_id"].map(id_maps["contester1_id"])
        df = df.drop(columns=["contester1_id"])

        df["Shot_Encoded"] = df["shottype"].map(shot_map)
        df = df.drop(columns=["shottype"])

        df["GameState_Encoded"] = df["gamestate"].map(gamestate_map)
        df = df.drop(columns=["gamestate"])

        df = df.drop(columns=["contester2_id"])

        df["Three_Encoded"] = df["three"].map(three_map)
        df = df.drop(columns=["three"])

        dribbles = df["dribblesbefore"]
        df["Dribbles (0, 1-2, 3+)"] = np.select(
            [dribbles < 1, dribbles.between(1, 2), dribbles > 2],
            [0, 2, 3],
            default=np.nan,
        )
        df = df.drop(columns=["dribblesbefore"])

        if "outcome" in df.columns:
            df["outcome"] = df["outcome"].astype(int)

        df = df.drop(columns=["shot_id"])

        # Softmax weighting returns an exponentially weighted average by raising Euler's number(e) to the power of  each index {0, 1, 2, 3} and dividing by the collective sum of the resulting values, balancing weighting of all float values in array with exponential prioritization of the more recent values

        # (e^0 / (e^0 + e^1 + e^2 + e^3)) = 0.032, (e^1 / (e^0 + e^1 + e^2 + e^3)) = 0.087, (e^2 / (e^0 + e^1 + e^2 + e^3)) = 0.237, (e^3 / (e^0 + e^1 + e^2 + e^3)) = 0.644

        new_floats = (
            df["closestdefapproach"].astype(str)
            .str.strip("{ ")
            .str.strip("} ")
            .str.split(",", expand=True)
            .astype(float)
        )
        df["def_approach"] = (
            new_floats[0] * 0.032
            + new_floats[1] * 0.087
            + new_floats[2] * 0.237
            + new_floats[3] * 0.644
        )
        df = df.drop(columns=["closestdefapproach"])

        # Calculation of shot's angle from basket using right triangle formed by vertical and horizontal distances from shooter to basket, multiplied by 100 to derive real angle value

        df["Shot_Angle"] = (
            np.arctan(df["locationy"].abs() / (df["locationx"] + 1e-5)) * 100
        ).abs()

        # Weighting of defensive pressure that weighs distance of closest defender if there is one more heavily while including number of shot contesters in the calculation of pressure

        df["def_pressure"] = (1 / df["closestdefdist"]) + df["num_contesters"]

        df["catch_shoot"] = (
            (df["Dribbles (0, 1-2, 3+)"] == 0) & (df["Shot_Encoded"] == "1")
        ).astype(int)

        df = df.drop(columns=[
            "offense", "Month_Encoded", "Season_Encoded", "defense",
            "distcont2", "defender", "locationy", "shooter", "closestdefdist",
        ])
        df["distcont1"] = df["distcont1"].fillna(15.0)
        df = df.drop(columns=["contester"])
        return df

    train = apply_changes(train)
    test = apply_changes(test)

    train.to_csv(train_out, index=False)
    test.to_csv(test_out, index=False)
    return train, test

In [4]:
from sklearn.ensemble import HistGradientBoostingClassifier

train_df, test_df = model_function()
shot_ids = pd.read_csv("testing.csv")["shot_id"]

X_train = train_df.drop(columns=["outcome"])
y_train = train_df["outcome"].astype(int)
X_test = test_df.drop(columns=["outcome"], errors="ignore")
X_test = X_test[X_train.columns]

# The encodings were stored as strings such as "0" and "1".
X_train = X_train.apply(pd.to_numeric, errors="coerce")
X_test = X_test.apply(pd.to_numeric, errors="coerce")

model = HistGradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.05,
    max_iter=500,
    max_leaf_nodes=63,
    min_samples_leaf=200,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=0,
)

model.fit(X_train, y_train)

# Column 1 is P(outcome == 1), treated here as the make probability.
make_prob = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "shot_id": shot_ids,
    "make_prob": make_prob,
})
submission.to_csv("submission.txt", index=False)

In [5]:
X = train_df.drop(columns=["outcome"]).apply(pd.to_numeric, errors="coerce")
y = train_df["outcome"].astype(int)

X_fit, X_val, y_fit, y_val = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

model = HistGradientBoostingClassifier(random_state=0)
model.fit(X_fit, y_fit)
val_prob = model.predict_proba(X_val)[:, 1]
score = log_loss(y_val, val_prob, labels=[0, 1])
print(score)

0.49672164765935883
